# Constructing a function

This section describes how to construct a `RealFunction` or `ComplexFunction`.

## Theory

When building functions as truncated Chebyshev series of order $N$ on an interval $[a;b]$:
```{math}
:label: cheb_series2
f(x) = \sum_{n=0}^{N} c_n T_n(x), \quad x \in [a, b]
\;,
```
there are two ways to define the coefficients $c_n$.
One approach is to project a known function, say $F$, onto the Chebyshev polynomials.
This approach is not used here, and the reader is referred to {cite:t}`trefethenApproximationTheoryApproximation2020` where the projection method is discussed in detail.
The second approach is to define $f$ as the interpolating function of $F$.
In other words, the coefficients $c_n$ are calculated so that $f(x)$ matches exactly $F(x)$ at $N+1$ points $x_n$.
In principle, the coefficients are obtained by solving the Vandermonde system:
```{math}
\begin{bmatrix}
T_0(x_0) & T_1(x_0) & T_2(x_0) & \dots \\
T_0(x_1) & T_1(x_1) & T_2(x_1) & \dots \\
T_0(x_2) & T_1(x_2) & T_2(x_2) & \dots \\
\vdots & \vdots & \vdots & \ddots
\end{bmatrix}
\begin{pmatrix}
c_0 \\
c_1 \\
c_2 \\
\vdots
\end{pmatrix}
=\begin{pmatrix}
F(x_0) \\
F(x_1) \\
F(x_2) \\
\vdots
\end{pmatrix}
\;.
```
Solving a dense linear system of size $N\times N$ has complexity $O(N^2)$.
This can become significant when $N$ is large.
A clever choice of interpolation points that reduces this cost significantly is to use the Chebyshev points of the second kind, as defined in {eq}`points2_std`.
In this way, the Vandermonde matrix becomes a DFT matrix (for Discrete Fourier Transform), and we can use the Fast Fourier Transform (FFT) to compute the coefficients with a complexity $O(N\log(N))$.

## Python API

### Real-valued interpolating functions

A Chebyshev series interpolating a known real-valued function $F$ can be easily created by using the class `RealFunction`.
Its constructor can be called by providing the function $F$, here a Gaussian function defined through a `lambda` function, together with the bounds $a$ and $b$ of the interval, here $[-2;2]$.

In [ ]:
import numpy as np
from cheby import RealFunction

f = RealFunction(lambda x: np.exp(-x**2), -2.0, 2.0)

We can then see how many coefficients $c_n$ have been defined

In [ ]:
print('Number of coefficients:', len(f.coef))

If the polynomial order $N$ is not specified, as in the example above, the constructor progressively increases the number of points, as powers of two ($2^4, 2^5, \dots, 2^{13}$), until the last 8 coefficients of the resulting series all fall below a relative tolerance of $10^{-14}$ of the largest coefficient.
It then trims the negligible trailing coefficients.
This adaptive behaviour means that a smooth function is represented with about as many coefficients as needed for machine-precision accuracy.

It is very useful to check the magnitude of the coefficients $c_n$, here with a log-scale:

In [ ]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_formats = ["svg", "pdf"]

plt.figure()
plt.semilogy(np.abs(f.coef), 'o-')
plt.xlabel(r'$n$')
plt.ylabel(r'$|c_n|$')
plt.grid()
plt.show()

We can see that for odd $n$ the coefficients $c_n$ are within machine precision of zero.
This should be expected since the function $F$ is even with respect to $x$ while the Chebyshev polynomials $T_n$ are odd for odd $n$.
We therefore only need the even $T_n$ (with $n$ even) to interpolate $F$.

Another crucial information in this graph is that $|c_n|$ decreases quickly when $n$ increasing (in fact, it is decreasing exponentially).
The functions $T_n$ are rapidly becoming less and less significant in our interpolation of $F$.
In this example, the magnitude of the last coefficient $c_{32}$ is close to $10^{-14}$.

A specific polynomial order for $f$ can be specified with the `N` argument, which disables the adaptive selection of the polynomial order.
Below, we specify an order of 50, meaning that 51 coefficients will be calculated:

In [ ]:
f = RealFunction(lambda x: np.exp(-x**2), -2.0, 2.0, 50)

print('Number of coefficients:', len(f.coef))

plt.figure()
plt.semilogy(np.abs(f.coef), 'o-')
plt.xlabel(r'$n$')
plt.ylabel(r'$|c_n|$')
plt.grid()
plt.show()

We can see that all the coefficients beyond $n=31$ are smaller than $10^{-16}$.
This is within machine precision of zero, and these coefficients can be safely neglected.

There are situations where the series in {eq}`cheb_series2` does not converge rapidly.
This is typically when the function $F$ is not smooth, for instance its value or some of its derivatives are discontinuous.
A simple example of this is the absolute value $F(x)=|x|$, which is continuous but its first derivative is discontinuous:

In [ ]:
f = RealFunction(lambda x: np.abs(x), -2.0, 2.0)

print('Number of coefficients:', len(f.coef))

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.semilogy(np.arange(0,len(f.coef),2), np.abs(f.coef[::2]), 'o-')
plt.xlabel(r'$n$ (only odd values)')
plt.ylabel(r'$|c_n|$')
plt.grid()

plt.subplot(1,2,2)
plt.loglog(np.arange(0,len(f.coef),2), np.abs(f.coef[::2]), 'o-')
plt.xlabel(r'$n$ (only odd values)')
plt.grid()

plt.show()

We can see that the magnitude of the coefficients $c_n$ doesn't decay exponentially with $n$.
In fact, looking at the log-log graph on the right, we see that they are decreasing only algebraically with $n$.
The constructor for `RealFunction` used the largest number of coefficients possible: $2^{13}+1$.
This illustrates that the ability of the Chebyshev series {eq}`cheb_series2` to accurately represent a function is strongly dependent on the smoothness of that solution.

### Complex-valued interpolating functions

For complex-valued function, one can use  `ComplexFunction` which uses a similar interface to `RealFunction`.

In [ ]:
from cheby import ComplexFunction

cf = ComplexFunction(lambda x: np.exp(1j * x), 0.0, 2 * np.pi)

print(len(cf.coef), 'coefficients of type', cf.coef.dtype)

plt.figure()
plt.semilogy(np.abs(cf.coef), 'o-')
plt.xlabel(r'$n$')
plt.ylabel(r'$|c_n|$')
plt.grid()
plt.show()

### Construction based on coefficients

A function can also be built directly from a coefficient vector, without sampling. This is how the results of arithmetic and calculus operations (see the following sections) are constructed internally, and it is also useful to build a function from coefficients obtained elsewhere:

In [ ]:
coef = np.array([1.0, 0.5, -0.25])
p = RealFunction(0.0, 1.0, coef)
print(p.coef)